# Notebook 1: Data Engineering & Preprocessing (Phase 1)



Run once, then load artifacts in other notebooks.



Pipeline (Cells 1-5):

1. Setup paths and imports

2. Chunked ratings ingestion + ID remapping

3. CSR ratings matrix + Truncated SVD (32 latent features)

4. Structured content attributes from genres/tags (Pazzani Table 10.1 style)

5. Save reusable outputs: `ratings_sparse.npz`, `latent_embeddings.npy`, `content_features.npy`

In [8]:
from pathlib import Path



import numpy as np

import pandas as pd

from scipy.sparse import csr_matrix, save_npz

from sklearn.decomposition import TruncatedSVD



RATINGS_PATH = Path("data/ml-32m/ratings.csv")

MOVIES_PATH = Path("data/ml-32m/movies.csv")

TAGS_PATH = Path("data/ml-32m/tags.csv")

OUTPUT_DIR = Path("data/processed_32m_phase1")



CHUNK_SIZE = 2_000_000

N_COMPONENTS = 32

SEED = 42

TOP_TAGS = 100



OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



print(f"Ratings path: {RATINGS_PATH.resolve()}")

print(f"Movies path:  {MOVIES_PATH.resolve()}")

print(f"Tags path:    {TAGS_PATH.resolve()}")

print(f"Output dir:   {OUTPUT_DIR.resolve()}")

Ratings path: C:\Users\acolumban\Desktop\Quantum-Enhanced-Smart-Shopping-Experience\QACF\data\ml-32m\ratings.csv
Movies path:  C:\Users\acolumban\Desktop\Quantum-Enhanced-Smart-Shopping-Experience\QACF\data\ml-32m\movies.csv
Tags path:    C:\Users\acolumban\Desktop\Quantum-Enhanced-Smart-Shopping-Experience\QACF\data\ml-32m\tags.csv
Output dir:   C:\Users\acolumban\Desktop\Quantum-Enhanced-Smart-Shopping-Experience\QACF\data\processed_32m_phase1


In [9]:
# Chunked loading + contiguous ID remapping

all_user_ids = set()

all_movie_ids = set()

total_rows = 0



for chunk in pd.read_csv(

    RATINGS_PATH,

    usecols=["userId", "movieId", "rating"],

    chunksize=CHUNK_SIZE

):

    total_rows += len(chunk)

    all_user_ids.update(chunk["userId"].unique().tolist())

    all_movie_ids.update(chunk["movieId"].unique().tolist())



unique_users = np.array(sorted(all_user_ids), dtype=np.int64)

unique_movies = np.array(sorted(all_movie_ids), dtype=np.int64)



user_to_idx = {uid: idx for idx, uid in enumerate(unique_users)}

movie_to_idx = {mid: idx for idx, mid in enumerate(unique_movies)}



print(f"Rows ingested: {total_rows:,}")

print(f"Unique users:  {len(unique_users):,}")

print(f"Unique movies: {len(unique_movies):,}")

Rows ingested: 32,000,204
Unique users:  200,948
Unique movies: 84,432


In [10]:
# CSR matrix + Truncated SVD (32 latent features)

row_parts, col_parts, val_parts = [], [], []



for chunk in pd.read_csv(

    RATINGS_PATH,

    usecols=["userId", "movieId", "rating"],

    chunksize=CHUNK_SIZE

):

    row_parts.append(chunk["userId"].map(user_to_idx).to_numpy(dtype=np.int32, copy=False))

    col_parts.append(chunk["movieId"].map(movie_to_idx).to_numpy(dtype=np.int32, copy=False))

    val_parts.append(chunk["rating"].to_numpy(dtype=np.float32, copy=False))



rows = np.concatenate(row_parts)

cols = np.concatenate(col_parts)

vals = np.concatenate(val_parts)



ratings_csr = csr_matrix(

    (vals, (rows, cols)),

    shape=(len(unique_users), len(unique_movies)),

    dtype=np.float32

)



svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=SEED)

latent_embeddings = svd.fit_transform(ratings_csr).astype(np.float32)



print(f"CSR shape: {ratings_csr.shape}")

print(f"CSR nnz:   {ratings_csr.nnz:,}")

print(f"Latent embeddings shape: {latent_embeddings.shape}")

CSR shape: (200948, 84432)
CSR nnz:   32,000,204
Latent embeddings shape: (200948, 32)


In [11]:
# Structured content attributes (genres + tags) and save outputs

movies = pd.read_csv(MOVIES_PATH, usecols=["movieId", "genres"])

movies = movies[movies["movieId"].isin(unique_movies)].copy()



movies["genres"] = movies["genres"].fillna("(no genres listed)")

genre_dummies = movies["genres"].str.get_dummies(sep="|")

genre_dummies.index = movies["movieId"].map(movie_to_idx)

genre_dummies = genre_dummies.sort_index()



tags = pd.read_csv(TAGS_PATH, usecols=["movieId", "tag"])

tags = tags[tags["movieId"].isin(unique_movies)].copy()

tags["tag"] = tags["tag"].astype(str).str.lower().str.strip()

tags = tags[tags["tag"] != ""]



if len(tags) > 0:

    top_tags = tags["tag"].value_counts().head(TOP_TAGS).index

    tags = tags[tags["tag"].isin(top_tags)]

    tags["value"] = 1

    tag_table = tags.pivot_table(index="movieId", columns="tag", values="value", aggfunc="max", fill_value=0)

    tag_table.index = tag_table.index.map(movie_to_idx)

    tag_table = tag_table.sort_index()

else:

    tag_table = pd.DataFrame()



full_index = np.arange(len(unique_movies), dtype=np.int32)

genre_dummies = genre_dummies.reindex(full_index, fill_value=0)

tag_table = tag_table.reindex(full_index, fill_value=0) if not tag_table.empty else pd.DataFrame(index=full_index)



content_table = pd.concat([genre_dummies, tag_table], axis=1).astype(np.float32)

content_features = content_table.to_numpy(dtype=np.float32, copy=False)



save_npz(OUTPUT_DIR / "ratings_sparse.npz", ratings_csr)

np.save(OUTPUT_DIR / "latent_embeddings.npy", latent_embeddings)

np.save(OUTPUT_DIR / "content_features.npy", content_features)



print(f"Genre attributes: {genre_dummies.shape[1]}")

print(f"Tag attributes:   {tag_table.shape[1]}")

print(f"Content matrix shape: {content_features.shape}")

print("Saved outputs:")

print(f"- {OUTPUT_DIR / 'ratings_sparse.npz'}")

print(f"- {OUTPUT_DIR / 'latent_embeddings.npy'}")

print(f"- {OUTPUT_DIR / 'content_features.npy'}")

Genre attributes: 20
Tag attributes:   100
Content matrix shape: (84432, 120)
Saved outputs:
- data\processed_32m_phase1\ratings_sparse.npz
- data\processed_32m_phase1\latent_embeddings.npy
- data\processed_32m_phase1\content_features.npy
